# PEN OIS FV Dashboard
**Central Bank Scenario Tool — BCRP Meeting Paths vs Market**

Loads the BCRP OIS strip, lets you define weighted rate-path scenarios, computes a fair value (FV), and visualises the discrepancy meeting-by-meeting.

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & USER CONFIGURATION
# ============================================================
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import warnings
warnings.filterwarnings('ignore')

# ── USER SETTINGS ──────────────────────────────────────────
DATA_FILE    = r'C:/Users/user/Downloads/vp.csv'  # raw OIS data
BCRP_RATE    = 4.75     # current BCRP overnight rate (%)
HIST_MONTHS  = 18       # months of history in the rate chart

# BCRP meeting dates — update each year
# Format: YYYY-MM-DD
MEETING_DATES_STR = [
    '2025-03-06', '2025-04-10', '2025-05-08',
    '2025-06-12', '2025-07-10', '2025-08-14',
    '2025-09-11', '2025-10-09', '2025-11-06',
    '2025-12-11', '2026-01-08', '2026-02-05',
]
N_MTG = 10   # number of meetings shown in dashboard
# ───────────────────────────────────────────────────────────

MEETINGS   = [pd.Timestamp(d) for d in MEETING_DATES_STR[:N_MTG]]
MTG_LABELS = [d.strftime('%b-%y') for d in MEETINGS]
TODAY      = pd.Timestamp.today().normalize()
_NODES     = [1, 30, 60, 90, 180, 210, 360]

print(f'Dashboard configured for {N_MTG} meetings: {MTG_LABELS[0]} → {MTG_LABELS[-1]}')
print(f'Current BCRP rate: {BCRP_RATE}%')

In [ ]:
# ============================================================
# CELL 2 — DATA LOADING & CONSTANT-MATURITY CURVE
# ============================================================

def _build_cmt(filepath, overnight_rate):
    """Load vp.csv and build daily constant-maturity OIS curve."""
    df = pd.read_csv(filepath)
    df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
    df['mat']  = pd.to_datetime(df['mat'],  format='%d/%m/%Y')
    df['ttm']  = (df['mat'] - df['date']).dt.days

    rows = []
    for dt, g in df.groupby('date'):
        idx = np.argsort(g['ttm'].values)
        d   = g['ttm'].values[idx]
        r   = g['rate'].values[idx]
        d, ui = np.unique(d, return_index=True)
        r  = r[ui]
        ir = np.interp(_NODES, d, r)
        rows.append({'date': dt, **{f'{n}d': float(v) for n, v in zip(_NODES, ir)}})

    df_c = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)
    df_c.insert(1, '1d', overnight_rate)   # anchor overnight to policy rate
    return df_c


def _sample_curve(overnight_rate, months=30):
    """Generate realistic sample OIS curve for demo when CSV is unavailable."""
    dates  = pd.date_range(end=TODAY, periods=months * 21, freq='B')
    n      = len(dates)
    rng    = np.random.default_rng(42)
    walk   = np.cumsum(rng.normal(0, 0.012, n))
    walk  -= walk[-1]
    base   = overnight_rate + walk
    df_c   = pd.DataFrame({'date': dates, '1d': base})
    offsets = {30: -0.08, 60: -0.16, 90: -0.27, 180: -0.41, 210: -0.47, 360: -0.62}
    for node, off in offsets.items():
        noise = np.cumsum(rng.normal(0, 0.004, n))
        noise -= noise.mean()
        df_c[f'{node}d'] = (base + off + noise * 0.4).clip(1.0, 9.0)
    return df_c


try:
    df_curve = _build_cmt(DATA_FILE, BCRP_RATE)
    print(f'✓  OIS data loaded:  {df_curve["date"].min().date()}  →  {df_curve["date"].max().date()}')
    print(f'   Observations: {len(df_curve)}')
except Exception as exc:
    print(f'⚠  Cannot load {DATA_FILE}\n   ({exc})')
    print('   Running on synthetic sample data.')
    df_curve = _sample_curve(BCRP_RATE)

df_curve.tail(3)

In [ ]:
# ============================================================
# CELL 3 — MARKET-IMPLIED PATH AT BCRP MEETINGS
# ============================================================

def curve_at_meetings(latest_row, meetings):
    """Interpolate OIS curve at each meeting date (TTM in days)."""
    d0   = pd.Timestamp(latest_row['date'])
    ttms = [max(1, (m - d0).days) for m in meetings]
    rts  = [float(latest_row[f'{n}d']) for n in _NODES]
    return [round(float(np.interp(t, _NODES, rts)), 4) for t in ttms]


def levels_to_moves(levels, start_rate):
    """Convert cumulative rate levels to meeting-by-meeting bps moves."""
    prev, moves = start_rate, []
    for lv in levels:
        moves.append(round((lv - prev) * 100, 1))
        prev = lv
    return moves


latest        = df_curve.iloc[-1]
mkt_levels    = curve_at_meetings(latest, MEETINGS)
mkt_moves_bps = levels_to_moves(mkt_levels, BCRP_RATE)

print(f'Market data as of: {pd.Timestamp(latest["date"]).strftime("%d %b %Y")}')
print(f'\n{"Meeting":<12} {"Rate (%)":>10}  {"Move (bps)":>12}')
print('-' * 38)
for lbl, lv, mv in zip(MTG_LABELS, mkt_levels, mkt_moves_bps):
    print(f'{lbl:<12} {lv:>10.3f}  {mv:>+12.1f}')

In [ ]:
# ============================================================
# CELL 4 — DEFAULT SCENARIOS & FV ENGINE
# ============================================================

def _sc(name, cut_meetings, n_mtg=N_MTG, bps=-25):
    """Build a scenario: -25 bps cut at the given (0-indexed) meeting positions."""
    moves = [0] * n_mtg
    for i in cut_meetings:
        if i < n_mtg:
            moves[i] = bps
    return {'name': name, 'weight': 0.0, 'moves': moves}


DEFAULT_SCENARIOS = [
    _sc('March',         [0]),
    _sc('April',         [1]),
    _sc('June',          [3]),
    _sc('July',          [4]),
    _sc('Mar/Jun',       [0, 3]),
    _sc('Jun/Sep',       [3, 6]),
    _sc('Mar/Jun/Sep',   [0, 3, 6]),
    _sc('Apr/Jun/Sep',   [1, 3, 6]),
    _sc('Mar/Apr',       [0, 1]),
    _sc('Sep',           [6]),
    _sc('Dec',           [9]),
    _sc('Hold',          []),
]

# Assign equal initial weights
_eq = round(100 / len(DEFAULT_SCENARIOS), 1)
for sc in DEFAULT_SCENARIOS:
    sc['weight'] = _eq
DEFAULT_SCENARIOS[-1]['weight'] = round(100 - _eq * (len(DEFAULT_SCENARIOS) - 1), 1)


# ── FV Engine ───────────────────────────────────────────────
def compute_fv(scenarios, mkt_lv, bcrp_rate):
    """
    Given a list of scenarios with weights, compute:
      - scenario-level rate paths
      - FV = weighted-average rate at each meeting
      - dispersion, Mkt-FV, etc.
    """
    n      = len(mkt_lv)
    sc_all = []
    total_w = 0.0

    for sc in scenarios:
        w   = sc['weight'] / 100.0
        mv  = sc['moves'][:n]
        lv, prev = [], bcrp_rate
        for m in mv:
            prev = round(prev + m / 100, 4)
            lv.append(prev)
        sc_all.append({'name': sc['name'], 'weight': sc['weight'], 'levels': lv})
        if w > 0:
            total_w += w

    fv_lv = []
    for i in range(n):
        if total_w > 0:
            wsum = sum(s['weight'] / 100.0 * s['levels'][i] for s in sc_all)
            fv_lv.append(round(wsum / total_w, 4))
        else:
            fv_lv.append(bcrp_rate)

    sc_disp = []
    for i in range(n):
        if total_w > 0:
            mean = fv_lv[i]
            var  = sum(s['weight'] / 100.0 * (s['levels'][i] - mean) ** 2
                       for s in sc_all) / total_w
            sc_disp.append(round(np.sqrt(var) * 100, 2))
        else:
            sc_disp.append(0.0)

    fv_mv  = levels_to_moves(fv_lv, bcrp_rate)
    mkt_mv = levels_to_moves(mkt_lv, bcrp_rate)
    wt_ev  = [round((fv - mk) * 100, 2) for fv, mk in zip(fv_lv, mkt_lv)]
    mkt_fv = [round((mk - fv) * 100, 2) for mk, fv in zip(mkt_lv, fv_lv)]
    score  = [round(e / d, 3) if d else 0 for e, d in zip(wt_ev, sc_disp)]

    return {
        'sc_levels': sc_all,
        'fv_levels': fv_lv,
        'fv_moves':  fv_mv,
        'mkt_levels': mkt_lv,
        'mkt_moves':  mkt_mv,
        'wt_ev':     wt_ev,
        'sc_disp':   sc_disp,
        'mkt_fv':    mkt_fv,
        'score':     score,
    }


print(f'Loaded {len(DEFAULT_SCENARIOS)} default scenarios.')

In [ ]:
# ============================================================
# CELL 5 — HTML TABLE RENDERER
# ============================================================

def _cell_bg(val, threshold=2.0):
    """Return inline style for a bps-move cell."""
    if val <= -threshold:  return 'background:#ffd6d6;'
    if val >=  threshold:  return 'background:#d6f5d6;'
    return ''


def render_strip_table(result, mtg_labels):
    n        = len(mtg_labels)
    sc_all   = result['sc_levels']
    fv_lv    = result['fv_levels']
    mkt_lv   = result['mkt_levels']
    mkt_fv   = result['mkt_fv']
    wt_ev    = result['wt_ev']
    sc_disp  = result['sc_disp']
    score    = result['score']

    col_w = 72

    css = f"""
    <style>
    .ois {{font-family:'Segoe UI',Arial,sans-serif;border-collapse:collapse;
           font-size:11px;width:100%;margin-top:6px;}}
    .ois th {{background:#1f2d40;color:#fff;padding:5px {col_w//8}px;
              text-align:center;white-space:nowrap;font-weight:600;letter-spacing:.3px;}}
    .ois td {{padding:3px 6px;text-align:center;border:1px solid #e4e8ec;
              white-space:nowrap;}}
    .ois tr:nth-child(even) td {{background:#f8fafc;}}
    .sc-name {{text-align:left!important;padding-left:10px!important;font-weight:500;}}
    .row-sep td {{border-top:2px solid #bbb!important;}}
    .row-mkt  td {{background:#e8f4e8!important;font-weight:700;}}
    .row-fv   td {{background:#e8eef8!important;font-weight:700;}}
    .row-diff td {{font-weight:700;}}
    .row-stat td {{color:#777;font-style:italic;}}
    </style>
    """

    hdr = ''.join(
        f'<th style="min-width:{col_w}px">{lbl}</th>'
        for lbl in mtg_labels[:n]
    )

    html = css + f"""
    <table class="ois">
    <thead>
      <tr>
        <th style="text-align:left;min-width:120px">Scenario</th>
        <th style="min-width:48px">Wt%</th>
        {hdr}
      </tr>
    </thead>
    <tbody>
    """

    # Scenario rows
    for sc in sc_all:
        cells = ''
        prev  = BCRP_RATE
        for i, lv in enumerate(sc['levels'][:n]):
            mv    = (lv - prev) * 100
            style = _cell_bg(mv)
            cells += f'<td style="{style}">{lv:.3f}</td>'
            prev = lv
        html += (
            f'<tr><td class="sc-name">{sc["name"]}</td>'
            f'<td>{sc["weight"]:.1f}</td>{cells}</tr>\n'
        )

    # Separator + Mkt row
    mkt_cells = ''.join(f'<td>{v:.3f}</td>' for v in mkt_lv[:n])
    html += (
        f'<tr class="row-sep row-mkt">'
        f'<td class="sc-name">Mkt</td><td>100</td>{mkt_cells}</tr>\n'
    )

    # FV row
    fv_cells = ''.join(f'<td>{v:.3f}</td>' for v in fv_lv[:n])
    html += (
        f'<tr class="row-fv">'
        f'<td class="sc-name">FV (wtd)</td><td>—</td>{fv_cells}</tr>\n'
    )

    # Mkt − FV row
    diff_cells = ''
    for v in mkt_fv[:n]:
        bg = 'background:#ffd6d6;' if v < -1.0 else ('background:#d6f5d6;' if v > 1.0 else '')
        diff_cells += f'<td style="{bg}font-weight:700">{v:+.1f}</td>'
    html += (
        f'<tr class="row-diff row-sep">'
        f'<td class="sc-name">Mkt − FV (bps)</td><td>—</td>{diff_cells}</tr>\n'
    )

    # Wtd EV row
    ev_cells = ''.join(f'<td>{v:+.2f}</td>' for v in wt_ev[:n])
    html += (
        f'<tr class="row-stat">'
        f'<td class="sc-name">Weighted EV (bps)</td><td>—</td>{ev_cells}</tr>\n'
    )

    # Dispersion row
    disp_cells = ''.join(f'<td>{v:.2f}</td>' for v in sc_disp[:n])
    html += (
        f'<tr class="row-stat">'
        f'<td class="sc-name">Sc. Dispersion (bps)</td><td>—</td>{disp_cells}</tr>\n'
    )

    # Score row
    sc_cells = ''.join(f'<td>{v:.3f}</td>' for v in score[:n])
    html += (
        f'<tr class="row-stat">'
        f'<td class="sc-name">Score (EV/Disp)</td><td>—</td>{sc_cells}</tr>\n'
    )

    html += '</tbody></table>'
    return html


print('Table renderer ready.')

In [ ]:
# ============================================================
# CELL 6 — CHART BUILDERS
# ============================================================

_SC_COLORS = [
    '#e53935', '#43a047', '#fb8c00', '#8e24aa', '#00acc1',
    '#6d4c41', '#e91e63', '#00897b', '#c0ca33', '#546e7a',
    '#1e88e5', '#795548',
]


# ── Chart 1: Meeting-by-meeting bar + cumulative ─────────────
def make_moves_chart(result, mtg_labels):
    n      = len(mtg_labels)
    mkt_mv = result['mkt_moves'][:n]
    fv_mv  = result['fv_moves'][:n]
    cm_mkt = list(np.cumsum(mkt_mv))
    cm_fv  = list(np.cumsum(fv_mv))

    fig = make_subplots(specs=[[{'secondary_y': True}]])

    fig.add_trace(go.Bar(
        x=mtg_labels[:n], y=mkt_mv,
        name='Mkt Move (bps)', marker_color='#4472C4',
        opacity=0.85, offsetgroup=1,
    ), secondary_y=False)

    fig.add_trace(go.Bar(
        x=mtg_labels[:n], y=fv_mv,
        name='FV Move (bps)', marker_color='#ED7D31',
        opacity=0.80, offsetgroup=2,
    ), secondary_y=False)

    fig.add_trace(go.Scatter(
        x=mtg_labels[:n], y=cm_mkt,
        mode='lines+markers', name='Cum Mkt',
        line=dict(color='#1a237e', width=2),
        marker=dict(size=6),
    ), secondary_y=True)

    fig.add_trace(go.Scatter(
        x=mtg_labels[:n], y=cm_fv,
        mode='lines+markers', name='Cum FV',
        line=dict(color='#bf360c', width=2, dash='dot'),
        marker=dict(size=6),
    ), secondary_y=True)

    fig.update_layout(
        title=dict(
            text='Meeting-by-Meeting Move (bps): Market vs Weighted FV + Cumulative',
            font=dict(size=13, color='#1a1a2e'), x=0.0,
        ),
        paper_bgcolor='white', plot_bgcolor='white',
        font=dict(color='#333', family='Arial', size=11),
        legend=dict(
            orientation='h', yanchor='bottom', y=1.02,
            xanchor='right', x=1, font=dict(size=10),
        ),
        barmode='group', bargap=0.18,
        height=340,
        margin=dict(l=55, r=55, t=60, b=40),
        xaxis=dict(gridcolor='#ececec', linecolor='#ccc', showgrid=False),
        yaxis=dict(
            title='bps (per meeting)',
            gridcolor='#ececec', zeroline=True, zerolinecolor='#aaa',
        ),
        yaxis2=dict(title='Cumulative bps', gridcolor='#ececec'),
    )
    return fig


# ── Chart 2: Realized OIS rate history + scenario paths ──────
def make_rate_chart(df_cv, result, meetings, hist_months=18):
    cutoff = TODAY - pd.DateOffset(months=hist_months)
    hist   = df_cv[df_cv['date'] >= cutoff].copy()
    n      = len(meetings)

    fig = go.Figure()

    # ── Historical lines ──
    line_specs = [
        ('90d',  'OIS 90d',  '#1a237e', 2.2, 1.0),
        ('180d', 'OIS 180d', '#5c6bc0', 1.4, 0.7),
        ('30d',  'OIS 30d',  '#90a4ae', 1.0, 0.55),
    ]
    for col, name, color, width, opacity in line_specs:
        if col in hist.columns:
            fig.add_trace(go.Scatter(
                x=hist['date'], y=hist[col],
                mode='lines', name=name,
                line=dict(color=color, width=width),
                opacity=opacity,
            ))

    # ── Scenario paths (thin colored lines from today forward) ──
    sc_all = result['sc_levels']
    for i, sc in enumerate(sc_all):
        w      = sc['weight']
        opacity = max(0.30, min(0.85, 0.35 + 0.55 * (w / 20)))
        dash    = 'solid' if w >= 10 else 'dot'
        fig.add_trace(go.Scatter(
            x=meetings[:n], y=sc['levels'][:n],
            mode='lines+markers',
            name=f'{sc["name"]} ({w:.1f}%)',
            line=dict(color=_SC_COLORS[i % len(_SC_COLORS)], width=1.3, dash=dash),
            marker=dict(size=5),
            opacity=opacity,
        ))

    # ── Market path ──
    fig.add_trace(go.Scatter(
        x=meetings[:n], y=result['mkt_levels'][:n],
        mode='lines+markers', name='Mkt Path',
        line=dict(color='#1565c0', width=2.0, dash='dash'),
        marker=dict(size=8, symbol='diamond', color='#1565c0'),
    ))

    # ── FV path (bold black) ──
    fig.add_trace(go.Scatter(
        x=meetings[:n], y=result['fv_levels'][:n],
        mode='lines+markers', name='FV (wtd avg)',
        line=dict(color='#000000', width=2.8),
        marker=dict(size=9, symbol='circle', color='black'),
    ))

    # ── Current rate reference line ──
    fig.add_hline(
        y=BCRP_RATE, line_dash='dash', line_color='#aaa', line_width=1,
        annotation_text=f'BCRP {BCRP_RATE}%',
        annotation_position='top right',
        annotation_font=dict(color='#888', size=10),
    )

    fig.update_layout(
        title=dict(
            text='PEN OIS — Realized Rate vs Scenario Paths',
            font=dict(size=14, color='#1a1a2e'), x=0.0,
        ),
        paper_bgcolor='white', plot_bgcolor='white',
        font=dict(color='#333', family='Arial', size=11),
        xaxis=dict(
            title='', gridcolor='#f0f0f0', linecolor='#ccc',
            showgrid=True, zeroline=False,
        ),
        yaxis=dict(
            title='Rate (%)', gridcolor='#f0f0f0', linecolor='#ccc',
            tickformat='.2f', showgrid=True, zeroline=False,
        ),
        legend=dict(
            orientation='v', yanchor='top', y=1.0,
            xanchor='left', x=1.01, font=dict(size=9),
            bgcolor='rgba(255,255,255,0.85)',
            bordercolor='#ddd', borderwidth=1,
        ),
        height=500,
        margin=dict(l=60, r=200, t=60, b=50),
        hovermode='x unified',
    )
    return fig


print('Chart builders ready.')

In [ ]:
# ============================================================
# CELL 7 — INTERACTIVE DASHBOARD
# ============================================================

import copy


# ── Build the widget grid from DEFAULT_SCENARIOS ─────────────
def build_row_widgets(sc):
    w_name = widgets.Text(
        value=sc['name'],
        layout=widgets.Layout(width='108px'),
    )
    w_wt = widgets.FloatText(
        value=sc['weight'],
        step=0.1,
        layout=widgets.Layout(width='60px'),
    )
    w_moves = [
        widgets.IntText(
            value=int(sc['moves'][i]) if i < len(sc['moves']) else 0,
            layout=widgets.Layout(width='52px'),
        )
        for i in range(N_MTG)
    ]
    return {'name': w_name, 'wt': w_wt, 'moves': w_moves}


row_ws = [build_row_widgets(sc) for sc in DEFAULT_SCENARIOS]


def read_scenarios():
    return [
        {
            'name':   r['name'].value,
            'weight': float(r['wt'].value),
            'moves':  [int(w.value) for w in r['moves']],
        }
        for r in row_ws
    ]


# ── Header row ───────────────────────────────────────────────
_HDR_STYLE = 'font-weight:700;font-size:11px;font-family:Arial;text-align:center;'
hdr_items = (
    [widgets.HTML(f'<div style="{_HDR_STYLE}width:108px;text-align:left">Scenario</div>')]
    + [widgets.HTML(f'<div style="{_HDR_STYLE}width:60px">Wt %</div>')]
    + [widgets.HTML(f'<div style="{_HDR_STYLE}width:52px">{lbl}</div>')
       for lbl in MTG_LABELS]
)
header_hbox = widgets.HBox(hdr_items)


def make_grid_row(rw):
    return widgets.HBox([rw['name'], rw['wt']] + rw['moves'])


grid_vbox = widgets.VBox([make_grid_row(rw) for rw in row_ws])


# ── Buttons ──────────────────────────────────────────────────
btn_update   = widgets.Button(
    description='Update Output',
    button_style='info',
    layout=widgets.Layout(width='140px', height='32px'),
)
btn_prices   = widgets.Button(
    description='Show Prices',
    button_style='',
    layout=widgets.Layout(width='120px', height='32px'),
)
btn_refresh  = widgets.Button(
    description='Refresh Market Data',
    button_style='success',
    layout=widgets.Layout(width='180px', height='32px'),
)
lbl_date = widgets.HTML(
    f'<span style="font-size:11px;color:#666;">&nbsp;'
    f'Last update: <b>{pd.Timestamp(latest["date"]).strftime("%d %b %Y")}</b></span>'
)


# ── Output areas ─────────────────────────────────────────────
out_note   = widgets.Output()   # info / subtitle line
out_table  = widgets.Output()   # repriced strip HTML table
out_charts = widgets.Output()   # Plotly charts


# ── Core update function ──────────────────────────────────────
def run_dashboard(_=None):
    scenarios = read_scenarios()
    result    = compute_fv(scenarios, mkt_levels, BCRP_RATE)
    total_w   = sum(s['weight'] for s in scenarios)

    with out_note:
        clear_output(wait=True)
        display(HTML(
            f'<p style="font-family:Arial;font-size:11px;color:#555;margin:0;">'
            f'Updated PEN OIS with {len(scenarios)} paths and {N_MTG} contracts. '
            f'Anchor: BCRP overnight rate ({BCRP_RATE}%). '
            f'Total scenario weight: <b>{total_w:.1f}%</b>. '
            f'Mode: rate vs Mkt.</p>'
        ))

    with out_table:
        clear_output(wait=True)
        display(HTML(render_strip_table(result, MTG_LABELS)))

    with out_charts:
        clear_output(wait=True)
        make_moves_chart(result, MTG_LABELS).show()
        make_rate_chart(df_curve, result, MEETINGS, HIST_MONTHS).show()


def refresh_market(_=None):
    """Re-read latest row from df_curve and rerun."""
    global mkt_levels, mkt_moves_bps, latest
    latest        = df_curve.iloc[-1]
    mkt_levels    = curve_at_meetings(latest, MEETINGS)
    mkt_moves_bps = levels_to_moves(mkt_levels, BCRP_RATE)
    lbl_date.value = (
        f'<span style="font-size:11px;color:#666;">&nbsp;'
        f'Last update: <b>{pd.Timestamp(latest["date"]).strftime("%d %b %Y")}</b></span>'
    )
    run_dashboard()


btn_update.on_click(run_dashboard)
btn_prices.on_click(run_dashboard)
btn_refresh.on_click(refresh_market)


# ── Layout assembly ───────────────────────────────────────────
title_w = widgets.HTML(
    '<h2 style="font-family:\'Segoe UI\',Arial,sans-serif;'
    'color:#1a237e;margin:0 0 4px 0;font-size:20px;">'
    'PEN OIS FV Dashboard</h2>'
    '<p style="font-family:Arial;font-size:11px;color:#888;margin:0 0 10px 0;">'
    'SOFR-style meeting-path tool for BCRP OIS strip · Anchor: overnight rate · '
    'Distribution: market OIS levels</p>'
)

top_bar   = widgets.HBox(
    [btn_refresh, lbl_date],
    layout=widgets.Layout(margin='0 0 8px 0', align_items='center'),
)
ctrl_row  = widgets.HBox(
    [btn_update, btn_prices],
    layout=widgets.Layout(margin='6px 0 0 0'),
)

editor_box = widgets.VBox(
    [header_hbox, grid_vbox, ctrl_row],
    layout=widgets.Layout(
        border='1px solid #d0d7de', padding='8px 10px',
        background='#fafbfc', border_radius='6px',
    ),
)

section_title = lambda t: widgets.HTML(
    f'<h4 style="font-family:Arial;color:#1a237e;margin:14px 0 4px 0;">'
    f'{t}</h4>'
)

dashboard = widgets.VBox([
    title_w,
    top_bar,
    editor_box,
    out_note,
    section_title('Repriced Strip  (rate %)'),
    out_table,
    section_title('Charts'),
    out_charts,
], layout=widgets.Layout(max_width='1200px'))

display(dashboard)
run_dashboard()   # initial render